In [2]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import pandas as pd
import numpy as np

# Verify configuration paths and package imports
from src.config import (
    RAW_DATA_PATH,
    DUCKDB_PATH,
    DE_COCK_OUTLIER_THRESHOLD,
    TARGET_COL
)

print(f"Data Target        : {RAW_DATA_PATH}")
print(f"File Exists        : {RAW_DATA_PATH.exists()}")
print(f"DuckDB Target Path : {DUCKDB_PATH}")

if RAW_DATA_PATH.exists():
    df_raw = pd.read_csv(RAW_DATA_PATH)
    print(f"Dataset Loaded     : {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
    display(df_raw.head(3))
else:
    print("Warning: RAW_DATA_PATH not found. Run the terminal download step!")

Data Target        : C:\Users\abine\Desktop\House Prediction DA\house-price-valuation\data\raw\AmesHousing.csv
File Exists        : True
DuckDB Target Path : C:\Users\abine\Desktop\House Prediction DA\house-price-valuation\data\housing.duckdb
Dataset Loaded     : 2,930 rows, 82 columns


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000


In [3]:
# Establish persistent DuckDB connection
con = duckdb.connect(str(DUCKDB_PATH))

# Ingest raw CSV directly into DuckDB's columnar storage
con.execute(f"""
    CREATE OR REPLACE TABLE raw_housing AS 
    SELECT * FROM read_csv_auto('{RAW_DATA_PATH.as_posix()}', header=True);
""")

# Verify ingestion counts via SQL
table_count = con.execute("SELECT COUNT(*) FROM raw_housing;").fetchone()[0]
table_cols = con.execute("DESCRIBE raw_housing;").fetchall()

print(f"Table 'raw_housing' successfully created in {DUCKDB_PATH.name}")
print(f"Ingested Rows    : {table_count:,}")
print(f"Total Columns    : {len(table_cols)}")

# Inspect the primary target variable distribution directly through SQL
con.execute("""
    SELECT 
        MIN(SalePrice) AS min_price,
        QUANTILE_CONT(SalePrice, 0.25) AS p25_price,
        MEDIAN(SalePrice) AS median_price,
        AVG(SalePrice)::INT AS mean_price,
        QUANTILE_CONT(SalePrice, 0.75) AS p75_price,
        MAX(SalePrice) AS max_price
    FROM raw_housing;
""").df()

Table 'raw_housing' successfully created in housing.duckdb
Ingested Rows    : 2,930
Total Columns    : 82


,min_price,p25_price,median_price,mean_price,p75_price,max_price
0,12789,129500.0,160000.0,180796,213500.0,755000


In [7]:
# SQL Audit for the official Dean De Cock outlier recommendation:
# Outliers: Gr Liv Area > 4000 sq ft (agricultural / abnormal scale builds)
outlier_audit_sql = f"""
    SELECT 
        "Order", 
        "PID", 
        "Neighborhood", 
        "Gr Liv Area" AS gr_liv_area, 
        "Sale Condition" AS sale_condition, 
        "SalePrice" AS sale_price
    FROM raw_housing
    WHERE "Gr Liv Area" > {DE_COCK_OUTLIER_THRESHOLD}
    ORDER BY "Gr Liv Area" DESC;
"""

df_outliers = con.execute(outlier_audit_sql).df()
print(f"Observations exceeding {DE_COCK_OUTLIER_THRESHOLD} sq ft: {len(df_outliers)}")
display(df_outliers)

# Audit transaction breakdown (Arm's-length 'Normal' vs Non-Normal sales)
condition_breakdown_sql = """
    SELECT 
        "Sale Condition" AS sale_condition, 
        COUNT(*) AS count, 
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_share,
        MEDIAN(SalePrice) AS median_price
    FROM raw_housing
    GROUP BY "Sale Condition"
    ORDER BY count DESC;
"""

display(con.execute(condition_breakdown_sql).df())

Observations exceeding 4000 sq ft: 5


,Order,PID,Neighborhood,gr_liv_area,sale_condition,sale_price
0,1499,908154235,Edwards,5642,Partial,160000
1,2181,908154195,Edwards,5095,Partial,183850
2,2182,908154205,Edwards,4676,Partial,184750
3,1761,528320050,NoRidge,4476,Abnorml,745000
4,1768,528351010,NoRidge,4316,Normal,755000


,sale_condition,count,pct_share,median_price
0,Normal,2413,82.35,159000.0
1,Partial,245,8.36,250000.0
2,Abnorml,190,6.48,129450.0
3,Family,46,1.57,144400.0
4,Alloca,24,0.82,149617.0
5,AdjLand,12,0.41,110000.0


In [8]:
# Cell 4: Materialize Curated Analytical Table in DuckDB
con.execute("""
    CREATE OR REPLACE TABLE housing_curated AS
    SELECT 
        "Order" AS order_id,
        "PID" AS parcel_id,
        "MS SubClass" AS ms_subclass,
        "MS Zoning" AS ms_zoning,
        "Lot Frontage" AS lot_frontage,
        "Lot Area" AS lot_area,
        "Street" AS street,
        "Alley" AS alley,
        "Lot Shape" AS lot_shape,
        "Land Contour" AS land_contour,
        "Utilities" AS utilities,
        "Lot Config" AS lot_config,
        "Land Slope" AS land_slope,
        "Neighborhood" AS neighborhood,
        "Condition 1" AS condition_1,
        "Condition 2" AS condition_2,
        "Bldg Type" AS bldg_type,
        "House Style" AS house_style,
        "Overall Qual" AS overall_qual,
        "Overall Cond" AS overall_cond,
        "Year Built" AS year_built,
        "Year Remod/Add" AS year_remod_add,
        "Roof Style" AS roof_style,
        "Roof Matl" AS roof_matl,
        "Exterior 1st" AS exterior_1st,
        "Exterior 2nd" AS exterior_2nd,
        "Mas Vnr Type" AS mas_vnr_type,
        "Mas Vnr Area" AS mas_vnr_area,
        "Exter Qual" AS exter_qual,
        "Exter Cond" AS exter_cond,
        "Foundation" AS foundation,
        "Bsmt Qual" AS bsmt_qual,
        "Bsmt Cond" AS bsmt_cond,
        "Bsmt Exposure" AS bsmt_exposure,
        "BsmtFin Type 1" AS bsmt_fin_type_1,
        "BsmtFin SF 1" AS bsmt_fin_sf_1,
        "BsmtFin Type 2" AS bsmt_fin_type_2,
        "BsmtFin SF 2" AS bsmt_fin_sf_2,
        "Bsmt Unf SF" AS bsmt_unf_sf,
        "Total Bsmt SF" AS total_bsmt_sf,
        "Heating" AS heating,
        "Heating QC" AS heating_qc,
        "Central Air" AS central_air,
        "Electrical" AS electrical,
        "1st Flr SF" AS first_flr_sf,
        "2nd Flr SF" AS second_flr_sf,
        "Low Qual Fin SF" AS low_qual_fin_sf,
        "Gr Liv Area" AS gr_liv_area,
        "Bsmt Full Bath" AS bsmt_full_bath,
        "Bsmt Half Bath" AS bsmt_half_bath,
        "Full Bath" AS full_bath,
        "Half Bath" AS half_bath,
        "Bedroom AbvGr" AS bedroom_abvgr,
        "Kitchen AbvGr" AS kitchen_abvgr,
        "Kitchen Qual" AS kitchen_qual,
        "TotRms AbvGrd" AS tot_rms_abv_grd,
        "Functional" AS functional,
        "Fireplaces" AS fireplaces,
        "Fireplace Qu" AS fireplace_qu,
        "Garage Type" AS garage_type,
        "Garage Yr Blt" AS garage_yr_blt,
        "Garage Finish" AS garage_finish,
        "Garage Cars" AS garage_cars,
        "Garage Area" AS garage_area,
        "Garage Qual" AS garage_qual,
        "Garage Cond" AS garage_cond,
        "Paved Drive" AS paved_drive,
        "Wood Deck SF" AS wood_deck_sf,
        "Open Porch SF" AS open_porch_sf,
        "Enclosed Porch" AS enclosed_porch,
        "3Ssn Porch" AS three_ssn_porch,
        "Screen Porch" AS screen_porch,
        "Pool Area" AS pool_area,
        "Pool QC" AS pool_qc,
        "Fence" AS fence,
        "Misc Feature" AS misc_feature,
        "Misc Val" AS misc_val,
        "Mo Sold" AS mo_sold,
        "Yr Sold" AS yr_sold,
        "Sale Type" AS sale_type,
        "Sale Condition" AS sale_condition,
        "SalePrice" AS sale_price
    FROM raw_housing
    WHERE "Gr Liv Area" <= 4000
      AND "Sale Condition" = 'Normal';
""")

# Confirm row count retained
curated_rows = con.execute("SELECT COUNT(*) FROM housing_curated;").fetchone()[0]
print(f"housing_curated table created: {curated_rows:,} records retained.")

# Inspect summary statistics of target variable
con.execute("""
    SELECT 
        MIN(sale_price) AS min_price,
        QUANTILE_CONT(sale_price, 0.25) AS p25_price,
        MEDIAN(sale_price) AS median_price,
        AVG(sale_price)::INT AS mean_price,
        QUANTILE_CONT(sale_price, 0.75) AS p75_price,
        MAX(sale_price) AS max_price,
        STDDEV(sale_price)::INT AS std_price
    FROM housing_curated;
""").df()

housing_curated table created: 2,412 records retained.


,min_price,p25_price,median_price,mean_price,p75_price,max_price,std_price
0,35000,129500.0,158750.0,175327,206225.0,625000,70006


In [9]:
# Cell 5: Missing Value Audit & Structural Null Identification
df_curated = con.execute("SELECT * FROM housing_curated").df()

# Calculate missingness percentage across all columns
null_counts = df_curated.isnull().sum()
null_pct = (null_counts / len(df_curated)) * 100
missing_report = (
    pd.DataFrame({"missing_count": null_counts, "missing_pct": null_pct})
    .query("missing_count > 0")
    .sort_values(by="missing_count", ascending=False)
)

print(f"Total features with missing values: {len(missing_report)}")
missing_report

Total features with missing values: 21


,missing_count,missing_pct
pool_qc,2403,99.626866
misc_feature,2316,96.019900
alley,2258,93.615257
fence,1910,79.187396
mas_vnr_type,1492,61.857380
fireplace_qu,1164,48.258706
lot_frontage,451,18.698176
garage_yr_blt,117,4.850746
garage_cond,117,4.850746
garage_qual,117,4.850746


In [10]:
# Cell 6: Domain-Aware Null Handling & Data Sanitation
df_clean = df_curated.copy()

# 1. Structural Categoricals: Replace NaN with 'None' (feature absent)
structural_cat_cols = [
    "pool_qc", "misc_feature", "alley", "fence", "fireplace_qu",
    "garage_type", "garage_finish", "garage_qual", "garage_cond",
    "bsmt_qual", "bsmt_cond", "bsmt_exposure", "bsmt_fin_type_1", "bsmt_fin_type_2",
    "mas_vnr_type"
]
for col in structural_cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("None")

# 2. Structural Numericals: Replace NaN with 0 (no area / count)
structural_num_cols = [
    "garage_yr_blt", "garage_cars", "garage_area",
    "bsmt_fin_sf_1", "bsmt_fin_sf_2", "bsmt_unf_sf", "total_bsmt_sf",
    "bsmt_full_bath", "bsmt_half_bath", "mas_vnr_area"
]
for col in structural_num_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# 3. Lot Frontage: Impute by Neighborhood Median (spatial proximity proxy)
df_clean["lot_frontage"] = df_clean.groupby("neighborhood")["lot_frontage"].transform(
    lambda s: s.fillna(s.median())
)
# Fallback for any neighborhood without a median
df_clean["lot_frontage"] = df_clean["lot_frontage"].fillna(df_clean["lot_frontage"].median())

# 4. Low-cardinality nominals: Mode imputation for rare administrative omissions
nominal_mode_cols = ["electrical"]
for col in nominal_mode_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Verify zero remaining unhandled missing values
remaining_nulls = df_clean.isnull().sum().sum()
print(f"Sanitation Complete. Remaining nulls in dataset: {remaining_nulls}")

# Re-write cleaned dataset to DuckDB as 'housing_clean'
con.execute("CREATE OR REPLACE TABLE housing_clean AS SELECT * FROM df_clean")
print(f"Persisted 'housing_clean' table into {DUCKDB_PATH.name} ({len(df_clean):,} rows, {df_clean.shape[1]} columns)")

Sanitation Complete. Remaining nulls in dataset: 0
Persisted 'housing_clean' table into housing.duckdb (2,412 rows, 82 columns)


In [11]:
# Cell 7: Explicit Feature Taxonomy Definition
# 1. Target and Identity columns
ID_COLS = ["order_id", "parcel_id"]
TARGET_COL = "sale_price"

# 2. Continuous scale features (sq footage, dimensions)
CONTINUOUS_FEATURES = [
    "lot_frontage", "lot_area", "mas_vnr_area", "bsmt_fin_sf_1", "bsmt_fin_sf_2",
    "bsmt_unf_sf", "total_bsmt_sf", "first_flr_sf", "second_flr_sf", 
    "low_qual_fin_sf", "gr_liv_area", "garage_area", "wood_deck_sf", 
    "open_porch_sf", "enclosed_porch", "three_ssn_porch", "screen_porch", 
    "pool_area", "misc_val"
]

# 3. Discrete count & temporal features
DISCRETE_FEATURES = [
    "year_built", "year_remod_add", "garage_yr_blt", "mo_sold", "yr_sold",
    "bsmt_full_bath", "bsmt_half_bath", "full_bath", "half_bath",
    "bedroom_abvgr", "kitchen_abvgr", "tot_rms_abv_grd", "fireplaces", "garage_cars"
]

# 4. Ordinal features with intrinsic hierarchy (Quality/Condition scales)
ORDINAL_MAPPINGS = {
    "exter_qual": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "exter_cond": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "bsmt_qual":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "bsmt_cond":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "heating_qc": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "kitchen_qual": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "fireplace_qu": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "garage_qual":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "garage_cond":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "pool_qc":      {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "bsmt_exposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "bsmt_fin_type_1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "bsmt_fin_type_2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "garage_finish":   {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "paved_drive":     {"N": 0, "P": 1, "Y": 2},
    "utilities":       {"ELO": 1, "NoSeWa": 2, "NoSewr": 3, "AllPub": 4},
    "land_slope":      {"Sev": 1, "Mod": 2, "Gtl": 3}
}

print(f"Taxonomy established:")
print(f" - Continuous features : {len(CONTINUOUS_FEATURES)}")
print(f" - Discrete features   : {len(DISCRETE_FEATURES)}")
print(f" - Ordinal features    : {len(ORDINAL_MAPPINGS)}")

Taxonomy established:
 - Continuous features : 19
 - Discrete features   : 14
 - Ordinal features    : 17


In [12]:
# Cell 8: Export Cleaned Artifact and Release Database Locks
from src.config import PROCESSED_DATA_DIR

PROCESSED_PARQUET_PATH = PROCESSED_DATA_DIR / "housing_clean.parquet"

# Save directly from DuckDB to high-performance Parquet format
con.execute(f"""
    COPY housing_clean TO '{PROCESSED_PARQUET_PATH.as_posix()}' (FORMAT PARQUET);
""")

# Verify export
assert PROCESSED_PARQUET_PATH.exists(), "Parquet export failed!"
parquet_size_mb = PROCESSED_PARQUET_PATH.stat().st_size / (1024 * 1024)
print(f"Exported clean analytical dataset to: {PROCESSED_PARQUET_PATH}")
print(f"File size: {parquet_size_mb:.2f} MB")

# Close DuckDB connection cleanly to avoid file locking on Windows
con.close()
print("DuckDB connection closed cleanly.")

Exported clean analytical dataset to: C:\Users\abine\Desktop\House Prediction DA\house-price-valuation\data\processed\housing_clean.parquet
File size: 0.17 MB
DuckDB connection closed cleanly.
